# Tier A application labs


This notebook is Part II‑A from
`smooth_manifolds_lie_groups_and_the_exponential_map.ipynb`:
the Bloch sphere ($\mathbb{CP}^{1}$), density operators on
$\mathbb{CP}^{d-1}$, unitary gates $\mathrm{U}(n)$, the symplectic
group $\mathrm{Sp}(2n,\mathbb{R})$, and the Fisher metric on a
statistical manifold. The wording below is that text. Examples
(M1)–(M3), charts and atlases, Lie groups and the exponential map,
and Tiers B and C stay in the mega notebook.

The code cells check those five labs. Lab A2 uses the Pauli
matrices defined in the A1 cell, as in the mega notebook. There
is no sibling demo.


In [ ]:
# Setup — foundations notebook (numpy / scipy / matplotlib only)
import numpy as np
import matplotlib.pyplot as plt
from numpy.linalg import norm, det, eigvalsh
from scipy.linalg import expm, logm

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.figsize": (5.5, 4.0), "font.size": 11})

print("numpy", np.__version__)


# Part II‑A — Application labs (full code)


Each lab below has the same shape: **why the object shows up**, the
**equations** that make it precise, a **coded check**, and a short
**takeaway** tying manifold / Lie group / exponential together.

> **Further reading (QI geometry).** I. Bengtsson and K. Życzkowski,
> *Geometry of Quantum States: An Introduction to Quantum
> Entanglement*, 2nd ed., Cambridge University Press, 2017. Chapter 4
> treats complex projective spaces; the Bloch sphere is $\mathbb{CP}^1$.


## A1. The Bloch sphere ($\mathbb{CP}^{1}$)


### Background


A **qubit** is the simplest quantum system with two levels (spin‑1/2,
ground/excited states of a superconducting circuit, polarisation of a
photon, …). Its pure states are unit vectors
$|\psi\rangle\in\mathbb{C}^2$, but physics does not care about an
overall complex factor: $|\psi\rangle$ and $\lambda|\psi\rangle$
($\lambda\neq 0$) describe the same ray. The space of such rays is the
complex projective line $\mathbb{CP}^{1}$. As a real smooth manifold
that space is ordinary $S^2$ — the **Bloch sphere**. That is why every
textbook picture of a qubit is a point on a sphere, not a vector in
$\mathbb{C}^2$.


### Geometry


$$
\mathbb{CP}^{1}\;\cong\; S^2.
$$

In the computational basis one may write

$$
|\psi\rangle
  =\cos\frac{\theta}{2}\,|0\rangle
  +e^{i\phi}\sin\frac{\theta}{2}\,|1\rangle,
\qquad
\theta\in[0,\pi],\quad \phi\in[0,2\pi),
$$

and the Bloch vector

$$
\mathbf{r}
  =(\sin\theta\cos\phi,\;\sin\theta\sin\phi,\;\cos\theta)
  \in S^2\subset\mathbb{R}^3
$$

satisfies $r_i=\langle\psi|\sigma_i|\psi\rangle$ with Pauli matrices
$\sigma_x,\sigma_y,\sigma_z$.

A Hamiltonian $H=\mathbf{n}\cdot\boldsymbol{\sigma}$ with
$|\mathbf{n}|=1$ generates the unitary one-parameter group

$$
U(t)=\exp(-i t H)\in\mathrm{U}(2),
$$

which rotates $\mathbf{r}(t)$ about the axis $\mathbf{n}$ at angular
speed $2$ (in units $\hbar=1$). So the **Lie algebra** (Hermitian
generators / anti‑Hermitian $X=-iH$) drives motion on the **manifold**
of states via the **exponential** in the **Lie group** $\mathrm{U}(2)$.


### Where it shows up


- **Single‑qubit gates** are rotations of $\mathbf{r}$ (or of the
  sphere itself under $\mathrm{SU}(2)$).
- **State tomography** recovers $\mathbf{r}$ from Pauli expectation
  values $\langle\sigma_i\rangle$.
- **NMR / ESR / superconducting control** design microwave pulses that
  steer $\mathbf{r}$ along chosen paths.
- **Decoherence** (not modelled in the pure‑state code below) pulls
  $\mathbf{r}$ inside the ball — which is the mixed‑state story of A2.


In [ ]:
# Bloch sphere: state <-> r, and precession under H = n · σ

PAULI = {
    "X": np.array([[0, 1], [1, 0]], dtype=complex),
    "Y": np.array([[0, -1j], [1j, 0]], dtype=complex),
    "Z": np.array([[1, 0], [0, -1]], dtype=complex),
}

def ket_from_bloch(theta, phi):
    return np.array(
        [np.cos(theta / 2), np.exp(1j * phi) * np.sin(theta / 2)],
        dtype=complex,
    )

def bloch_from_ket(psi):
    psi = psi / norm(psi)
    # remove global phase so <0|psi> is real nonnegative when possible
    if abs(psi[0]) > 1e-14:
        psi = psi * np.exp(-1j * np.angle(psi[0]))
    x = 2 * np.real(np.conj(psi[0]) * psi[1])
    y = 2 * np.imag(np.conj(psi[0]) * psi[1])
    z = np.abs(psi[0]) ** 2 - np.abs(psi[1]) ** 2
    return np.array([x, y, z])

def hamiltonian(n):
    n = np.asarray(n, dtype=float)
    n = n / norm(n)
    return n[0] * PAULI["X"] + n[1] * PAULI["Y"] + n[2] * PAULI["Z"]

theta0, phi0 = 0.7, 0.3
r0 = bloch_from_ket(ket_from_bloch(theta0, phi0))
n_axis = np.array([0.0, 0.0, 1.0])  # rotate about z
H = hamiltonian(n_axis)

ts = np.linspace(0, 2 * np.pi, 80)
path = []
for t in ts:
    psi_t = expm(-1j * t * H) @ ket_from_bloch(theta0, phi0)
    path.append(bloch_from_ket(psi_t))
path = np.array(path)

# Analytic: rotation about z leaves z fixed, rotates (x,y)
assert np.allclose(path[:, 2], r0[2], atol=1e-8)

fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(111, projection="3d")
u, v = np.mgrid[0 : 2 * np.pi : 40j, 0 : np.pi : 20j]
ax.plot_surface(
    np.cos(u) * np.sin(v),
    np.sin(u) * np.sin(v),
    np.cos(v),
    alpha=0.08,
    color="gray",
)
ax.plot(path[:, 0], path[:, 1], path[:, 2], "C0-", lw=2, label="orbit")
ax.scatter(*r0, color="C3", s=40, label="start")
ax.set_title(r"Bloch precession under $H=Z$")
ax.legend(fontsize=9)
plt.show()

print("start Bloch r =", r0)
print("||r(t)|| stays 1: max deviation",
      np.max(np.abs(norm(path, axis=1) - 1)))


**Takeaway box — A1.**  
Manifold: $\mathbb{CP}^{1}\cong S^2$ (pure qubit states). Lie group
acting: $\mathrm{U}(2)$ (or $\mathrm{SU}(2)$). Lie algebra:
anti‑Hermitian matrices $i\mathfrak{su}(2)$ / Hermitian Hamiltonians.
Exponential: $U(t)=\exp(-iHt)$.


## A2. $\mathbb{CP}^{d-1}$ and density operators


### Background


Not every quantum system is a qubit, and not every state is pure.
A $d$-level system (qutrit, qudit, truncated oscillator, …) has pure
states that are rays in $\mathbb{C}^d$ — points of the complex
projective space $\mathbb{CP}^{d-1}$. When the system is entangled with
an environment, or we simply lack full knowledge, the correct object is
a **density operator** $\rho$: a Hermitian, positive‑semidefinite
matrix with $\operatorname{Tr}\rho=1$. Pure states sit at the extreme
points ($\rho=|\psi\rangle\langle\psi|$); everything else is a
statistical mixture.


### Geometry


Pure states of a $d$-level system are points of $\mathbb{CP}^{d-1}$.
The set of all density operators is a **convex body** in the real
vector space of Hermitian matrices of trace one. Its extreme points
are the pure states (rank‑one projectors). Fixed‑rank subsets are
smooth manifolds in the relative interior; the full set has corners
(a stratified space / manifold with corners). For $d=2$ that body is
the **Bloch ball** $|\mathbf{r}|\le 1$; the sphere $|\mathbf{r}|=1$ is
exactly A1.

A useful scalar invariant is the **purity**

$$
\mathcal{P}(\rho)=\operatorname{Tr}(\rho^2),
\qquad
\frac{1}{d}\le\mathcal{P}(\rho)\le 1,
$$

with $\mathcal{P}=1$ if and only if $\rho$ is pure.


### Where it shows up


- **Open quantum systems** and noise channels act as maps on $\rho$,
  typically shrinking purity.
- **Quantum information geometry** (Bures metric, quantum Fisher
  information) lives on full‑rank density operators — distances that
  quantify how distinguishable two states are.
- **Tomography and certification** reconstruct $\rho$ or check that a
  prepared state is close to a target pure state.
- **Entanglement** for bipartite systems is diagnosed on the joint
  $\rho_{AB}$; reduced states $\rho_A=\operatorname{Tr}_B\rho_{AB}$
  are generically mixed even when $\rho_{AB}$ is pure.

We check the geometry numerically for a qubit (Bloch ball) and a
qutrit purity slice.


In [ ]:
# Density operators: Bloch ball (qubit) and purity for random qutrit states

def random_density(d, rng, pure=False):
    if pure:
        v = rng.normal(size=d) + 1j * rng.normal(size=d)
        v /= norm(v)
        return np.outer(v, np.conj(v))
    A = rng.normal(size=(d, d)) + 1j * rng.normal(size=(d, d))
    rho = A @ A.conj().T
    return rho / np.trace(rho)

rng = np.random.default_rng(1)
bloch_pts = []
for _ in range(400):
    rho = random_density(2, rng)
    # r_i = Tr(rho sigma_i)
    r = np.array(
        [np.real(np.trace(rho @ PAULI[s])) for s in ("X", "Y", "Z")]
    )
    bloch_pts.append(r)
bloch_pts = np.array(bloch_pts)

fig = plt.figure(figsize=(10, 4))
ax1 = fig.add_subplot(121, projection="3d")
ax1.scatter(
    bloch_pts[:, 0], bloch_pts[:, 1], bloch_pts[:, 2],
    s=8, alpha=0.5, c=norm(bloch_pts, axis=1), cmap="viridis",
)
ax1.set_title("Random qubit states (Bloch ball)")
ax1.set_xlim(-1, 1); ax1.set_ylim(-1, 1); ax1.set_zlim(-1, 1)

pur_mixed = []
pur_pure = []
for _ in range(300):
    rm = random_density(3, rng, pure=False)
    rp = random_density(3, rng, pure=True)
    pur_mixed.append(np.real(np.trace(rm @ rm)))
    pur_pure.append(np.real(np.trace(rp @ rp)))

ax2 = fig.add_subplot(122)
ax2.hist(pur_mixed, bins=30, alpha=0.7, label="mixed (Wishart)")
ax2.axvline(1.0, color="C1", ls="--", lw=2, label="pure (= 1)")
ax2.set_xlabel(r"purity $\mathrm{Tr}(\rho^2)$")
ax2.set_title("Qutrit purity")
ax2.legend(fontsize=9)
plt.tight_layout()
plt.show()

assert np.all(norm(bloch_pts, axis=1) <= 1 + 1e-10)
assert np.allclose(pur_pure, 1.0)
print("max |r| for mixed qubits:", np.max(norm(bloch_pts, axis=1)))
print("pure qutrit purity min/max:", np.min(pur_pure), np.max(pur_pure))
print("mixed qutrit purity range:", np.min(pur_mixed), np.max(pur_mixed))


**Takeaway box — A2.**  
Pure states: manifold $\mathbb{CP}^{d-1}$. Mixed states: convex set
of density operators (stratified by rank). Geometry of distinguishability
(Bures, quantum Fisher) lives on the interior of full-rank states —
see Bengtsson–Życzkowski for the Riemannian picture.


## A3. Unitary gates $\mathrm{U}(n)$ and Hermitian generators


### Background


Closed quantum evolution preserves inner products: if
$|\psi(0)\rangle$ is normalised, so is $|\psi(t)\rangle$, and relative
phases between components are the entire dynamical content. The maps
that do that are **unitary** matrices. Every gate in a digital quantum
circuit, and every continuous drive under a time‑dependent
Hamiltonian, is (ideally) an element of $\mathrm{U}(n)$ — or of the
special unitary group $\mathrm{SU}(n)$ once an irrelevant global phase
is fixed. For $k$ qubits one takes $n=2^k$.


### Geometry


$$
\mathrm{U}(n)=\{U\in\mathrm{M}_n(\mathbb{C}):U^\dagger U=I\}
$$

is a compact Lie group of real dimension $n^2$. Its Lie algebra
$\mathfrak{u}(n)$ consists of **anti‑Hermitian** matrices
$X^\dagger=-X$. Writing $X=-iH$ with $H=H^\dagger$, every one-parameter
subgroup is

$$
U(t)=\exp(-i t H).
$$

The traceless subalgebra $\mathfrak{su}(n)$ (dimension $n^2-1$)
generates $\mathrm{SU}(n)$. Controllability questions in quantum
control ask which subgroup you can reach by switching among a finite
list of Hamiltonians — a Lie‑algebra generation problem.


### Where it shows up


- **Gate synthesis / compilation:** approximate a target $U_*$ by a
  product of native exponentials $\exp(-i t_j H_j)$.
- **Hamiltonian simulation** (module 05): write
  $\exp(-i t(A+B+\cdots))$ as a product of simpler exponentials
  (Trotter–Suzuki) — only after the exponential map of Part I is in
  place.
- **Pulse‑level control:** shaped microwave / laser drives implement
  time‑ordered exponentials of $H(t)$.


In [ ]:
# U(n): exp(-iH) is unitary when H is Hermitian; dimension check

def random_hermitian(n, rng):
    A = rng.normal(size=(n, n)) + 1j * rng.normal(size=(n, n))
    return (A + A.conj().T) / 2

rng = np.random.default_rng(2)
H = random_hermitian(4, rng)
U = expm(-1j * H)
print("||U^† U - I|| =", norm(U.conj().T @ U - np.eye(4)))
print("|det U| =", abs(det(U)))

# Real dimension of U(n) is n^2; of su(n) (traceless anti-Hermitian) is n^2-1
n = 4
# sample: n^2 independent real parameters for Hermitian H
print(f"real dim of u({n}) / Hermitian generators: {n * n}")
print(f"real dim of su({n}) (traceless): {n * n - 1}")


**Takeaway box — A3.**  
Lie group: $\mathrm{U}(n)$ (quantum gates). Lie algebra: $i$ times
Hermitian matrices. Exponential: time evolution / gate synthesis.


## A4. The symplectic group $\mathrm{Sp}(2n,\mathbb{R})$


### Background


Position and momentum $(q_1,p_1,\ldots,q_n,p_n)$ live in a $2n$‑dimensional
**phase space**. Classical Hamiltonian mechanics cares about more than
ordinary volumes: it cares about the **symplectic** area form that
makes Hamilton's equations covariant. Linear maps that preserve that
form are the **symplectic** matrices. The same group appears in
quantum optics and continuous‑variable (CV) quantum information: beam
splitters, phase shifters, and squeezers act as Gaussian unitaries
whose classical shadow on means and covariances is exactly a
symplectic transformation.


### Geometry


Phase space $\mathbb{R}^{2n}$ carries the standard symplectic matrix
$J$ ($J^\top=-J$, $J^2=-I$). A real matrix $S$ is **symplectic** when

$$
S^\top J S = J.
$$

The set of all such $S$ is the Lie group $\mathrm{Sp}(2n,\mathbb{R})$.
For $n=1$ one has $\mathrm{Sp}(2,\mathbb{R})\cong\mathrm{SL}(2,\mathbb{R})$.
Quadratic Hamiltonians generate one‑parameter subgroups by the matrix
exponential in the Lie algebra $\mathfrak{sp}(2n,\mathbb{R})$.

A Gaussian state's covariance matrix $\Sigma$ transforms as
$\Sigma\mapsto S\Sigma S^\top$. Physical states obey an uncertainty
relation that can be written, with $\hbar=1$, as
$\Sigma+\mathrm{i}J/2\ge 0$.


### Where it shows up


- **Classical mechanics:** linear canonical transformations; small
  oscillations; stability of fixed points.
- **Quantum optics / CV QI:** squeezing, two‑mode entanglement
  (EPR‑type states), Gaussian boson sampling backends.
- **Metrology:** squeezed vacuum improves phase sensitivity beyond
  shot noise within the Gaussian sector.

> **Further reading.** M. de Gosson, *Symplectic Geometry and Quantum
> Mechanics*, Birkhäuser, 2006, Ch. 2 (the symplectic group).


In [ ]:
# Sp(2,R) ≅ SL(2,R): check S.T @ J @ S = J, and act on a covariance ellipse

def J_matrix(n):
    J = np.zeros((2 * n, 2 * n))
    for i in range(n):
        J[2 * i, 2 * i + 1] = 1
        J[2 * i + 1, 2 * i] = -1
    return J

J = J_matrix(1)
# A one-parameter subgroup: squeezing / rotation generators in sp(2,R)
# K_rot = [[0,-1],[1,0]] generates rotations; K_sq = [[0,1],[1,0]] squeeze

def sp_rot(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])

def sp_squeeze(r):
    return np.array([[np.exp(r), 0.0], [0.0, np.exp(-r)]])

for name, S in [("rot", sp_rot(0.4)), ("sq", sp_squeeze(0.5)),
                ("prod", sp_rot(0.3) @ sp_squeeze(0.2))]:
    err = norm(S.T @ J @ S - J)
    print(f"{name}: ||S^T J S - J|| = {err:.2e}, det={det(S):.6f}")

# Covariance ellipse of a Gaussian: Σ = S S^T starting from identity
S = sp_rot(0.5) @ sp_squeeze(0.7)
Sigma = S @ S.T
# Williamson / uncertainty: Σ + i (ħ/2) J ≥ 0 with ħ=1 → Σ + i J/2 ≥ 0
W = Sigma + 1j * J / 2
evals = eigvalsh(W)
print("eigenvalues of Σ + i J/2 (should be >= 0):", np.real(evals))

fig, ax = plt.subplots()
ts = np.linspace(0, 2 * np.pi, 200)
circle = np.stack([np.cos(ts), np.sin(ts)], axis=0)
ellipse = S @ circle
ax.plot(circle[0], circle[1], "k--", lw=1, label="vacuum")
ax.plot(ellipse[0], ellipse[1], "C0-", lw=2, label="S · vacuum")
ax.set_aspect("equal")
ax.set_title(r"Symplectic image of the unit circle in $\mathbb{R}^2$")
ax.legend(fontsize=9)
plt.show()


**Takeaway box — A4.**  
Lie group: $\mathrm{Sp}(2n,\mathbb{R})$ (linear canonical /
Gaussian CV gates). Lie algebra: $\mathfrak{sp}(2n,\mathbb{R})$
(Hamiltonian matrices for quadratic forms). Exponential: Gaussian
unitaries' classical shadow.


## A5. Statistical manifolds and the Fisher metric


### Background


Estimation problems ask: given samples from a probability distribution
that depends on an unknown parameter $\theta$, how accurately can
$\theta$ be recovered? Differential geometry enters because the family
of distributions itself is a curved space, and there is a natural
notion of distance that matches statistical distinguishability.


### Geometry


A smooth family of probability densities $\{p_\theta\}$ parametrised
by $\theta\in\Theta\subset\mathbb{R}^m$ is a **statistical manifold**.
The **Fisher information matrix**

$$
g_{ij}(\theta)
  =\mathbb{E}_\theta
   \bigl[\partial_i\log p_\theta\;\partial_j\log p_\theta\bigr]
$$

is a Riemannian metric on $\Theta$ (Rao, 1945). The Cramér–Rao bound
says that the covariance of an unbiased estimator is at least
$g(\theta)^{-1}$: more Fisher information means tighter possible
error bars. Geodesics and distances derived from $g$ organise
asymptotic statistics.

For a Bernoulli random variable with success probability $p\in(0,1)$,

$$
g(p)=\frac{1}{p(1-p)},
$$

which blows up at the endpoints — extreme probabilities are easier to
tell apart from their neighbours than $p\approx 1/2$ is.


### Where it shows up


- **Classical statistics:** asymptotic efficiency, exponential families,
  natural gradients in machine learning (Amari).
- **Quantum metrology:** the **quantum Fisher information** on density
  operators bounds how well a phase or field can be estimated; squeezed
  and entangled probes aim to raise that information per resource.
- **Thermodynamics / inference:** geometries on spaces of Gibbs states
  connect to response and fluctuation formulae.

> **Further reading.** C. R. Rao, “Information and the accuracy
> attainable in the estimation of statistical parameters,” *Bull.
> Calcutta Math. Soc.* **37** (1945) 81–91. S. Amari,
> *Differential-Geometrical Methods in Statistics*, Springer, 1985.
> S. Amari and H. Nagaoka, *Methods of Information Geometry*, AMS,
> 2000 (Ch. 2–3).


In [ ]:
# Bernoulli(p) as a 1-D statistical manifold; Fisher metric g(p)=1/(p(1-p))

def fisher_bernoulli(p):
    return 1.0 / (p * (1.0 - p))

ps = np.linspace(0.05, 0.95, 200)
g = fisher_bernoulli(ps)

# Fisher arc length from 0.2 to 0.8: integral sqrt(g) dp = Delta arcosh-like
# For Bernoulli, the Fisher distance is 2|arcsin(sqrt(p1))-arcsin(sqrt(p0))|
def fisher_dist_bernoulli(p0, p1):
    return 2 * np.abs(np.arcsin(np.sqrt(p1)) - np.arcsin(np.sqrt(p0)))

p0, p1 = 0.2, 0.8
# Numerical integrate sqrt(g)
integ = np.trapezoid(np.sqrt(fisher_bernoulli(np.linspace(p0, p1, 2000))),
                     np.linspace(p0, p1, 2000))
analytic = fisher_dist_bernoulli(p0, p1)
print("Fisher distance 0.2→0.8 numeric:", integ)
print("Fisher distance analytic:      ", analytic)

fig, ax = plt.subplots()
ax.plot(ps, g, "C0-", lw=2)
ax.set_xlabel(r"$p$")
ax.set_ylabel(r"$g(p)=1/(p(1-p))$")
ax.set_title("Fisher metric on the Bernoulli family")
plt.show()


**Takeaway box — A5.**  
Manifold: parameter space of a smooth statistical model. Metric:
Fisher–Rao. Quantum thermodynamics and quantum estimation use the
quantum Fisher information on density-operator manifolds (Amari–Nagaoka,
Ch. on quantum systems).
